# QUBO Formulation for Multi-Target Data Association

Demonstrates the MTDA to QUBO conversion (Fraunhofer FKIE arXiv:2110.08346):
- Cost matrix from log-likelihood ratios
- Binary variable encoding
- Row/column constraint penalties
- Auto-calibrated penalty terms

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quantum_common.visualization.styles import apply_publication_style

apply_publication_style()

from quantum_mht.formulation.cost_matrix import CostMatrixBuilder
from quantum_mht.formulation.association_variables import AssociationVariables
from quantum_mht.formulation.constraint_encoder import ConstraintEncoder
from quantum_mht.formulation.mtda_qubo_builder import MTDAQuboBuilder

In [ ]:
# Synthetic tracking scenario: 5 targets, 8 measurements
rng = np.random.default_rng(42)
n_targets, n_meas = 5, 8

true_positions = rng.uniform(10, 90, size=(n_targets, 2))
measurements = np.vstack([
    true_positions + rng.normal(0, 2, size=(n_targets, 2)),  # True detections
    rng.uniform(0, 100, size=(n_meas - n_targets, 2)),      # Clutter
])
covariances = np.array([np.eye(2) * 4.0 for _ in range(n_targets)])

# Visualize
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(*true_positions.T, s=100, c='blue', marker='o', label='Tracks', zorder=5)
ax.scatter(*measurements[:n_targets].T, s=60, c='green', marker='^', label='True detections')
ax.scatter(*measurements[n_targets:].T, s=40, c='red', marker='x', label='Clutter')
for i in range(n_targets):
    ax.annotate(f'T{i}', true_positions[i], fontsize=9)
ax.set_title(f'{n_targets} Targets, {n_meas} Measurements')
ax.legend()
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
plt.show()

In [ ]:
# Build cost matrix
cost_builder = CostMatrixBuilder(gate_threshold=20.0)
cost_matrix, gate_mask = cost_builder.build_with_gating_mask(
    true_positions, measurements, covariances,
)

print(f"Cost matrix shape: {cost_matrix.shape}")
print(f"Gated entries: {gate_mask.sum()} / {gate_mask.size}")

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(cost_matrix, cmap='viridis', aspect='auto')
ax.set_xlabel('Measurement index')
ax.set_ylabel('Track index')
ax.set_title('Cost Matrix (log-likelihood ratios)')
plt.colorbar(im, ax=ax)
plt.show()

In [ ]:
# Build QUBO
builder = MTDAQuboBuilder()
qubo_result = builder.build_from_cost_matrix(cost_matrix, gate_mask)

print(f"QUBO variables: {qubo_result.num_variables}")
print(f"Auto-calibrated penalty: {qubo_result.penalty:.2f}")
print(f"Q matrix entries: {len(qubo_result.Q)}")

# Variable breakdown
vars = qubo_result.variables
n_assign = n_targets * n_meas
n_missed = n_targets if vars.include_missed else 0
n_fa = n_meas if vars.include_false_alarm else 0
print(f"\nVariable breakdown:")
print(f"  Assignment x_{{i,j}}: {n_assign}")
print(f"  Missed detection: {n_missed}")
print(f"  False alarm: {n_fa}")
print(f"  Total: {n_assign + n_missed + n_fa}")